In [110]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mutual_info_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction import DictVectorizer


In [1]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

In [4]:
df = pd.read_csv(data)
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [6]:
df.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='object')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   object 
 1   industry                  4760 non-null   object 
 2   employment_status         4806 non-null   object 
 3   location                  4792 non-null   object 
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 351.7+ KB


In [88]:
categorical = df.select_dtypes(include=['object']).columns
categorical = categorical.to_list()
numerical = df.select_dtypes(include=['float64', 'int64']).columns.drop('converted', errors='ignore')
numerical = numerical.to_list()
print("Categorical columns:", categorical)
print("Numerical columns:", numerical)

Categorical columns: ['lead_source', 'industry', 'employment_status', 'location']
Numerical columns: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


In [89]:
df[categorical].isna().sum()
df[categorical] = df[categorical].fillna('NA')

In [90]:
df[numerical].isna().sum()
df[numerical] = df[numerical].fillna(0.0)

In [91]:
df.isna().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [92]:
df['industry'].mode()

0    technology
Name: industry, dtype: object

In [93]:
df[numerical].corr()


,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


In [94]:
target = [
    # 'annual_income', 'number_of_courses_viewed', 
    'interaction_count','lead_score'
]

for col in numerical:
    for i in target:
        if col == i:
            continue
        corr = df[col].corr(df[i])
        print(f"Correlation between '{i}' and '{col}': {corr:.4f}")


Correlation between 'interaction_count' and 'annual_income': 0.1228
Correlation between 'lead_score' and 'annual_income': 0.2295
Correlation between 'interaction_count' and 'number_of_courses_viewed': 0.7216
Correlation between 'lead_score' and 'number_of_courses_viewed': 0.7572
Correlation between 'lead_score' and 'interaction_count': 0.9157
Correlation between 'interaction_count' and 'lead_score': 0.9157


In [95]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [96]:
df_full_train.shape, df_train.shape, df_val.shape, df_test.shape

((4000, 9), (3000, 9), (1000, 9), (1000, 9))

In [97]:
df_train.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
4702,paid_ads,finance,employed,asia,65980.0,2,3,0.37,1
2155,events,finance,employed,south_america,71662.0,0,0,0.09,0
289,referral,NA,employed,asia,40220.0,2,5,0.55,1
1822,organic_search,finance,employed,north_america,88352.0,2,2,0.45,0
3442,events,technology,employed,north_america,89316.0,1,5,0.00,1


In [98]:
mutual_score = {}
for c in categorical:
    
    mutual_info = mutual_info_score(df_train.converted, (df_train[c]))
    mutual_score[c] = mutual_info

sorted(mutual_score.items(), key = lambda x: x[1], reverse = True)

[('lead_source', 0.03434605791617973),
 ('employment_status', 0.018965604906982875),
 ('industry', 0.0019945636715935217),
 ('location', 0.001042763579701525)]

In [128]:
def train(df, y, C):
    dicts = df[categorical + numerical].to_dict(orient = 'records')

    dv = DictVectorizer(sparse = False)
    X_train = dv.fit_transform(dicts)

    model = LogisticRegression(
        solver='liblinear', 
        C=1.0, 
        max_iter=1000, 
        random_state=42
    )
    model.fit(X_train, y)

    return dv, model

In [129]:
df.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='object')

In [130]:
categorical

['lead_source', 'industry', 'employment_status', 'location']

In [131]:
numerical

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [132]:
df_train.columns

Index(['lead_source', 'industry', 'employment_status', 'location',
       'annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='object')

In [133]:
def predict(df, dv, model):
    dicts = df[categorical + numerical].to_dict(orient = 'records')

    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]

    return y_pred

In [134]:
dv, model = train(df_train, df_train.converted, C = 1)

In [135]:
y_pred = predict(df_val, dv, model)
y_pred

array([0.61725691, 0.44364108, 0.60433996, 0.64562246, 0.52214159,
       0.77885722, 0.60698076, 0.54481903, 0.76770971, 0.52765495,
       0.75038798, 0.6470054 , 0.56857815, 0.51044645, 0.59921425,
       0.46620745, 0.62786484, 0.69167143, 0.69082953, 0.59101344,
       0.49964695, 0.55590822, 0.65303933, 0.6079306 , 0.7231576 ,
       0.51616805, 0.56141276, 0.57206269, 0.67043904, 0.55599224,
       0.63613322, 0.70337215, 0.45463513, 0.61897288, 0.65250689,
       0.42775649, 0.63943096, 0.62315373, 0.6602988 , 0.51736304,
       0.53774147, 0.76074014, 0.48326071, 0.74151288, 0.63662375,
       0.41194601, 0.45371292, 0.6417333 , 0.43851314, 0.87060984,
       0.72468973, 0.75643941, 0.5296636 , 0.66736678, 0.50444495,
       0.57644782, 0.72318139, 0.72497254, 0.6507741 , 0.628958  ,
       0.7217776 , 0.59231588, 0.70824617, 0.61451576, 0.49691434,
       0.74445314, 0.52773381, 0.61402574, 0.55979831, 0.59024431,
       0.69368703, 0.72962242, 0.76111566, 0.60744404, 0.52501

In [136]:
y_val = df_val.converted.values
y_val

array([1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0,
       1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
       0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1,
       0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1,
       1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0,
       1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1,
       0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0,
       0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0,
       1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0,
       1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1,
       1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0,
       0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1,

In [137]:
round(accuracy_score(y_val, y_pred >= 0.5), 2)

0.65

In [138]:
def accuracy_with_features(df, features):

    dicts = df[features].to_dict(orient='records')
    dv = DictVectorizer(sparse=False)

    X_train = dv.fit_transform(dicts)
    X_val = dv.transform(df_val[features].to_dict(orient='records'))

    model = LogisticRegression(
        solver='liblinear',
        C=1.0,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, df.converted)
    y_pred = model.predict_proba(X_val)[:, 1]
    return accuracy_score(y_val, y_pred >= 0.5)


features = categorical + numerical
baseline_accuracy = accuracy_with_features(df_train, features)

feature_elimination = {}
for feature in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    accuracy_without_feature = accuracy_with_features(
        df_train, [column for column in features if column != feature]
    )

    feature_elimination[feature] = {
        'accuracy': accuracy_without_feature,
        'difference': baseline_accuracy - accuracy_without_feature
    }

baseline_accuracy, feature_elimination

(0.645,
 {'lead_source': {'accuracy': 0.642, 'difference': 0.0030000000000000027},
  'number_of_courses_viewed': {'accuracy': 0.643,
   'difference': 0.0020000000000000018},
  'interaction_count': {'accuracy': 0.601, 'difference': 0.04400000000000004}})

In [139]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    
    dv, model = train(df_train, df_train.converted, C = C)

    y_pred = predict(df_val, dv, model)
    accuracy = ((y_pred >= 0.5) == y_val).mean()

    accuracy = accuracy_score(y_val, y_pred >= 0.5)
    print(f'{C}: {accuracy}')

1e-06: 0.645
1e-05: 0.645
0.0001: 0.645
0.001: 0.645


In [140]:
def accuracy_with_features(df, features):

    dicts = df[features].to_dict(orient='records')
    dv = DictVectorizer(sparse=False)

    X_train = dv.fit_transform(dicts)
    X_val = dv.transform(df_val[features].to_dict(orient='records'))

    model = LogisticRegression(
        solver='liblinear',
        C=1.0,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, df.converted)
    y_pred = model.predict_proba(X_val)[:, 1]
    return accuracy_score(y_val, y_pred >= 0.5)


features = categorical + numerical
baseline_accuracy = accuracy_with_features(df_train, features)

feature_elimination = {}
for feature in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    # accuracy_without_feature = accuracy_with_features(
    #     df_train, [column for column in features if column != feature]
    # )
    for col in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
        if col != feature:
            accuracy_without_feature = accuracy_with_features(df_train, col)

        feature_elimination[feature] = {
            'accuracy': accuracy_without_feature,
            'difference': baseline_accuracy - accuracy_without_feature
        }

baseline_accuracy, feature_elimination

TypeError: Series.to_dict() got an unexpected keyword argument 'orient'